In [ ]:
import numpy as np
import matplotlib.pyplot as plt

class Stimulus(object):
    def __init__(
        self,
        EI: np.ndarray, # (N_laps, L)
        PI: np.ndarray, # (N_laps, L)
    ):
        if EI.shape != PI.shape:
            raise ValueError("EI and PI must have the same shape.")
        self.EI = EI
        self.PI = PI

class ContinuousAttractorGraph(object):
    """This class represents a continuous attractor neural network graph."""
    def __init__(
        self,
        N: int,
        L: int,
        alpha_E: float = 0.3,
        Tk: float = 0.1,
    ):
        """Initialize `ContinuousAttractorGraph` Class

        Parameters
        ----------
        N : int
            The number of discrete attractors
        L : int
            The number of bins for the continuous variables.
        """
         
        self.N = N # Number of attractors
        self.L = L # Number of spatial bins
        self.E = np.ones((N, L)) # Energy matrix
        self.F = np.zeros((N, N, L-1)) # Forward probability
        self.J0 = np.zeros((N, N, L)) # Jump probability
        self.Jm = np.zeros((N, N, L)) # Modification factor for the jump probability
        self.A = np.zeros((N, N, L)) # Activation History
        self.S = np.zeros((N, L, 2)) # Stimuli Matrix
        self.S[:, :, 0] = np.tile(np.arange(L), (N, 1)) # Location-specific inputs
        
        self.alpha_E = alpha_E # Cross-day learning rate
        self.Tk = Tk # Noise level
        
        self._init_E()
        self._init_F()
        self._init_J0()
        
    def _init_E(self):
        """Initialize the first spatial map."""
        for pre_train_day in range(26):
            self.E[0, :] = self.SFER(self.E[0, :], np.ones(self.L), alpha=self.alpha_E)
    
    def _init_F(self):
        """Initialize the forward probability matrix F."""
        self.F[0, 0, :] = 1.0
    
    def _init_J0(self):
        """Initialize the jump probability matrix J0."""
        self.update_J0_day()
        
    def boltzmann(self, E: np.ndarray, Tk: float ) -> np.ndarray:
        """Compute the Boltzmann distribution for given energies.

        Parameters
        ----------
        E : np.ndarray
            Energy values.
        Tk : float
            Temperature parameter, or noise level.

        Returns
        -------
        np.ndarray
            Boltzmann distribution.
        """
        return np.exp(-E/Tk)
    
    def SFER(self, E: np.ndarray, A: np.ndarray, alpha: float) -> np.ndarray:
        assert E.shape == A.shape, "E and A must have the same shape."
        assert 0 <= alpha <= 1, "alpha must be in [0, 1]."
        return (0.99-alpha) * E + alpha * (1-A) + 0.01
    
    def update_E_day(self):
        self.E = self.SFER(self.E, self.A, alpha=self.alpha_E)
    
    def update_J0_day(self):
        """Update the jump probability matrix J0 based on the current energy matrix E."""
        P = self.boltzmann(self.E, Tk=self.Tk)
        for i in range(self.L):
            self.J0[:, :, i] = np.tile(((1-P[:, i])/(self.N-1))[:, np.newaxis], (1, self.N))
            self.J0[:, :, i][np.diag_indices(self.N)] = P[:, i]

CAG = ContinuousAttractorGraph(N=5, L=100)
J0 = CAG.J0
np.sum(J0, axis=1)  # Each slice along axis 1 should sum to 1

array([[1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1.],
       [1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1.],
       [1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1.